In [1]:
import pandas as pd
import numpy as np
import faiss
import os
from sentence_transformers import SentenceTransformer
from collections import defaultdict

# Set working directory
os.chdir('/Users/yaminiumarani/Desktop/rag_thesis_project')
print("Working directory:", os.getcwd())

# Load everything
df = pd.read_csv('data/github_issues_final.csv', low_memory=False)
model = SentenceTransformer('all-MiniLM-L6-v2')
index = faiss.read_index('data/baseline_faiss.index')

print("Dataset:", df.shape)
print("FAISS index:", index.ntotal, "vectors")
print("All loaded!")


Working directory: /Users/yaminiumarani/Desktop/rag_thesis_project


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dataset: (15955, 37)
FAISS index: 15955 vectors
All loaded!


In [2]:
# Enhanced author authority scoring
# Adding: reporter vs contributor distinction
# and cross-issue consistency

author_frequency = defaultdict(int)
author_resolution_count = defaultdict(int)
author_issues_set = defaultdict(set)  # track unique issues per author
reporter_set = set(df['user_login'].dropna().unique())  # people who only report

for idx, row in df.iterrows():
    answers = []
    for i in range(6):
        author = row.get(f'answers_{i}_author')
        body = row.get(f'answers_{i}_body')
        if pd.notna(author) and pd.notna(body):
            answers.append((i, author))
    
    for pos, (i, author) in enumerate(answers):
        # Count total appearances
        author_frequency[author] += 1
        
        # Track unique issues this author has contributed to
        author_issues_set[author].add(idx)
        
        # If last answer in closed issue = likely resolver
        # NOTE: compare POSITION in the answer list, not the raw slot index,
        # so issues with gaps (e.g. a filtered bot reply) credit the correct author
        if pos == len(answers) - 1:
            author_resolution_count[author] += 1

# Build enhanced dataframe
author_df2 = pd.DataFrame({
    'author': list(author_frequency.keys()),
    'frequency': list(author_frequency.values()),
    'resolutions': [author_resolution_count[a] for a in author_frequency.keys()],
    'unique_issues': [len(author_issues_set[a]) for a in author_frequency.keys()],
})

# Anonymised accounts excluded before any normalisation, so no derived
# quantity is scaled against an author absent from the final table
author_df2 = author_df2[author_df2['author'] != 'ghost'].reset_index(drop=True)

# Flag pure reporters — appear as issue creators but rarely as answer providers
author_df2['is_pure_reporter'] = (
    author_df2['author'].isin(reporter_set) & 
    (author_df2['frequency'] < 3)
)

# Cross issue consistency score
# Authors who contribute across many unique issues are more authoritative
author_df2['consistency_score'] = (
    author_df2['unique_issues'] / author_df2['unique_issues'].max()
)

# Enhanced authority score
# frequency(30%) + resolutions(40%) + consistency(30%)
# pure reporters get a penalty
author_df2['authority_score'] = (
    0.3 * (author_df2['frequency'] / author_df2['frequency'].max()) +
    0.4 * (author_df2['resolutions'] / author_df2['resolutions'].max()) +
    0.3 * author_df2['consistency_score']
)

# Apply penalty to pure reporters
author_df2.loc[author_df2['is_pure_reporter'], 'authority_score'] *= 0.2

# Normalise final score to 0-1
author_df2['authority_score'] = (
    author_df2['authority_score'] / author_df2['authority_score'].max()
)

author_df2 = author_df2.sort_values('authority_score', ascending=False).reset_index(drop=True)

print("=== TOP 10 ENHANCED AUTHORITY SCORES ===")
print(author_df2.head(10).to_string())
print(f"\nTotal pure reporters penalised: {author_df2['is_pure_reporter'].sum()}")
print(f"Total valid authors: {len(author_df2)}")

=== TOP 10 ENHANCED AUTHORITY SCORES ===
                author  frequency  resolutions  unique_issues  is_pure_reporter  consistency_score  authority_score
0              ocornut        296           92            132             False           1.000000         1.000000
1              sharkdp        218           62             98             False           0.742424         0.713238
2               lervag        184           71             86             False           0.651515         0.690637
3  magento-engcom-team        230           54             75             False           0.568182         0.638345
4            Conduitry        118           45             83             False           0.628788         0.503883
5            akien-mga        100           39             74             False           0.560606         0.439098
6     oliviertassinari        127           26             64             False           0.484848         0.387214
7              shilman         

In [3]:
# Save enhanced authority scores
author_df2.to_csv('data/author_authority_scores.csv', index=False)
print("Enhanced authority scores saved! ✅")

# Quick sanity check - look up a known author
test_author = 'ocornut'
score = author_df2[author_df2['author'] == test_author]['authority_score'].values[0]
print(f"\nAuthority score for '{test_author}': {score:.4f}")

# Look up a random first time commenter
low_auth = author_df2[author_df2['is_pure_reporter'] == True].tail(1)
print(f"Authority score for pure reporter '{low_auth['author'].values[0]}': {low_auth['authority_score'].values[0]:.4f}")

Enhanced authority scores saved! ✅

Authority score for 'ocornut': 1.0000
Authority score for pure reporter 'Bartman0': 0.0007


In [4]:
# Label relevance scoring
# Boosts issues whose labels match the query intent

label_keywords = {
    'bug': ['bug', 'error', 'crash', 'fail', 'broken', 'not working', 
            'incorrect', 'wrong', 'issue', 'problem'],
    'enhancement': ['feature', 'request', 'improve', 'add', 'support', 
                   'allow', 'enable', 'implement'],
    'documentation': ['doc', 'documentation', 'readme', 'guide', 
                     'tutorial', 'example', 'explain'],
    'good first issue': ['beginner', 'simple', 'easy', 'starter'],
    'help wanted': ['help', 'assist', 'support', 'need']
}

def get_label_relevance_score(query, label):
    if pd.isna(label):
        return 0.0
    
    query_lower = query.lower()
    label_lower = str(label).lower()
    
    # Direct label match in query
    if label_lower in query_lower:
        return 1.0
    
    # Check keyword matches
    for category, keywords in label_keywords.items():
        if category in label_lower:
            if any(kw in query_lower for kw in keywords):
                return 0.7
    
    return 0.1  # small base score

# Test it
test_cases = [
    ("window resize not working correctly", "bug"),
    ("window resize not working correctly", "enhancement"),
    ("add dark mode support", "enhancement"),
    ("add dark mode support", "bug"),
    ("documentation missing for API", "documentation"),
]

print("=== LABEL RELEVANCE SCORE TESTS ===")
for query, label in test_cases:
    score = get_label_relevance_score(query, label)
    print(f"Query: '{query}' | Label: '{label}' → Score: {score}")

=== LABEL RELEVANCE SCORE TESTS ===
Query: 'window resize not working correctly' | Label: 'bug' → Score: 0.7
Query: 'window resize not working correctly' | Label: 'enhancement' → Score: 0.1
Query: 'add dark mode support' | Label: 'enhancement' → Score: 0.7
Query: 'add dark mode support' | Label: 'bug' → Score: 0.1
Query: 'documentation missing for API' | Label: 'documentation' → Score: 1.0


In [5]:
# Build author authority lookup dictionary for fast access
authority_lookup = dict(zip(author_df2['author'], author_df2['authority_score']))

def get_author_authority(author):
    if pd.isna(author):
        return 0.0
    return authority_lookup.get(str(author), 0.05)  # unknown authors get low score

def dynamic_weighted_retrieve(query, top_k=5):
    
    # Step 1: Get initial candidates from FAISS (retrieve more than needed)
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding.astype(np.float32), top_k * 3)
    
    # Step 2: Score each candidate with dynamic weights
    scored_results = []
    
    for dist, idx in zip(distances[0], indices[0]):
        row = df.iloc[idx]
        
        # Semantic similarity score (convert distance to similarity)
        semantic_score = 1 / (1 + dist)
        
        # Label relevance score
        label_score = get_label_relevance_score(query, row.get('labels_0_name'))
        
        # Author authority score (best author in the thread)
        author_scores = []
        for i in range(6):
            author = row.get(f'answers_{i}_author')
            author_scores.append(get_author_authority(author))
        best_author_score = max(author_scores) if author_scores else 0.0
        
        # Dynamic weighted final score (Dw formula from your proposal)
        # Semantic(50%) + Author authority(30%) + Label relevance(20%)
        final_score = (
            0.5 * semantic_score +
            0.3 * best_author_score +
            0.2 * label_score
        )
        
        scored_results.append({
            'idx': idx,
            'title': row['title'],
            'label': row.get('labels_0_name'),
            'best_author': max(
                [(row.get(f'answers_{i}_author'), 
                  get_author_authority(row.get(f'answers_{i}_author'))) 
                 for i in range(6)],
                key=lambda x: x[1]
            )[0],
            'semantic_score': round(semantic_score, 4),
            'label_score': round(label_score, 4),
            'author_score': round(best_author_score, 4),
            'final_score': round(final_score, 4),
            'combined_text': str(row['combined_text'])
        })
    
    # Step 3: Re-rank by final weighted score
    scored_results = sorted(scored_results, key=lambda x: x['final_score'], reverse=True)
    
    return scored_results[:top_k]

# Test it!
query = "window resize not working correctly"
print(f"Query: '{query}'")
print("="*60)

results = dynamic_weighted_retrieve(query, top_k=5)
for r in results:
    print(f"\nRank {results.index(r)+1}")
    print(f"Title: {r['title']}")
    print(f"Label: {r['label']} | Best Author: {r['best_author']}")
    print(f"Scores → Semantic: {r['semantic_score']} | Author: {r['author_score']} | Label: {r['label_score']} | Final: {r['final_score']}")
    print("-"*40)

Query: 'window resize not working correctly'

Rank 1
Title: Bug when resizing window from borders (with io.ConfigResizeWindowsFromEdges=true)
Label: bug | Best Author: ocornut
Scores → Semantic: 0.5317000150680542 | Author: 1.0 | Label: 0.7 | Final: 0.7057999968528748
----------------------------------------

Rank 2
Title: Rectangle Doesn't Resize Windows
Label: bug | Best Author: rxhanson
Scores → Semantic: 0.5440999865531921 | Author: 0.3506 | Label: 0.7 | Final: 0.5171999931335449
----------------------------------------

Rank 3
Title: Dragging windows between screens makes them resize
Label: bug | Best Author: rxhanson
Scores → Semantic: 0.5432999730110168 | Author: 0.3506 | Label: 0.7 | Final: 0.5169000029563904
----------------------------------------

Rank 4
Title: Overlapping of the thirds 
Label: bug | Best Author: rxhanson
Scores → Semantic: 0.5317999720573425 | Author: 0.3506 | Label: 0.7 | Final: 0.5110999941825867
----------------------------------------

Rank 5
Title: 1/6

In [6]:
# Direct comparison: baseline vs dynamic weighted

def baseline_retrieve(query, top_k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    distances, indices = index.search(query_embedding.astype(np.float32), top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'rank': i + 1,
            'title': df['title'].iloc[idx],
            'label': df['labels_0_name'].iloc[idx],
            'author': df['answers_0_author'].iloc[idx]
        })
    return results

# Test with same query
query = "window resize not working correctly"

baseline = baseline_retrieve(query)
dynamic = dynamic_weighted_retrieve(query)

print(f"Query: '{query}'")
print("\n{:<5} {:<55} {:<20}".format("Rank", "BASELINE", "DYNAMIC WEIGHTED"))
print("="*100)

for i in range(5):
    b = baseline[i]
    d = dynamic[i]
    print(f"\n#{i+1}")
    print(f"  Baseline : {b['title'][:70]} | Author: {b['author']}")
    print(f"  Dynamic  : {d['title'][:70]} | Author: {d['best_author']} (score: {d['final_score']:.4f})")

Query: 'window resize not working correctly'

Rank  BASELINE                                                DYNAMIC WEIGHTED    

#1
  Baseline : [Feature Request] Allow resizeTo() (and other resize functionality) to | Author: ayushmanchhabra
  Dynamic  : Bug when resizing window from borders (with io.ConfigResizeWindowsFrom | Author: ocornut (score: 0.7058)

#2
  Baseline : Rectangle Doesn't Resize Windows | Author: rxhanson
  Dynamic  : Rectangle Doesn't Resize Windows | Author: rxhanson (score: 0.5172)

#3
  Baseline : Dragging windows between screens makes them resize | Author: rxhanson
  Dynamic  : Dragging windows between screens makes them resize | Author: rxhanson (score: 0.5169)

#4
  Baseline : Able to change the window height even if IsResizable is false but Exte | Author: nan
  Dynamic  : Overlapping of the thirds  | Author: rxhanson (score: 0.5111)

#5
  Baseline : Resizing a window doesn't always update the layout | Author: sdutoit
  Dynamic  : 1/6th size calculation for 

In [7]:
# Test with two more queries
queries = [
    "authentication token expiring too quickly",
    "dark mode not applying to all components"
]

for query in queries:
    baseline = baseline_retrieve(query)
    dynamic = dynamic_weighted_retrieve(query)
    
    print(f"\nQuery: '{query}'")
    print("="*80)
    for i in range(3):
        b = baseline[i]
        d = dynamic[i]
        print(f"\n#{i+1}")
        print(f"  Baseline : {b['title'][:65]} | Author: {b['author']}")
        print(f"  Dynamic  : {d['title'][:65]} | Author: {d['best_author']} (score: {d['final_score']:.4f})")


Query: 'authentication token expiring too quickly'

#1
  Baseline : Clarify cookie session refresh/invalidation | Author: JoelSpeed
  Dynamic  : Allow to configure JWT expiration | Author: lauriejim (score: 0.3630)

#2
  Baseline : Token-refresh failing for performance tests | Author: malvikach
  Dynamic  : Clarify cookie session refresh/invalidation | Author: JoelSpeed (score: 0.3310)

#3
  Baseline : getIdToken throws "auth/user-token-expired" if refresh token is r | Author: hsubox76
  Dynamic  : Handle case when JWT token verification fails | Author: Martinsos (score: 0.3152)

Query: 'dark mode not applying to all components'

#1
  Baseline : [Blazor] Some of the default UI styling is not accessible in dark | Author: javiercn
  Dynamic  : <Typography variant="button"> is always dark | Author: oliviertassinari (score: 0.3814)

#2
  Baseline : Deployed links sometimes start in dark mode | Author: yangshun
  Dynamic  : Deployed links sometimes start in dark mode | Author: yangshun (sc

In [8]:
import ollama

# Test Ollama connection
response = ollama.chat(
    model='llama3.2:1b',
    messages=[{'role': 'user', 'content': "Say 'Ollama connection successful!' and nothing else."}]
)

print(response['message']['content'])

Ollama connection successful!


In [9]:
import time

# Test smaller model speed
start = time.time()

response = ollama.chat(
    model='llama3.2:1b',
    messages=[{'role': 'user', 'content': "Say 'Ollama connection successful!' and nothing else."}]
)

end = time.time()
print(response['message']['content'])
print(f"Response time: {round(end - start, 2)} seconds")

Ollama connection successful!
Response time: 0.43 seconds


In [10]:
import ollama, time

GEN_OPTIONS = {'temperature': 0, 'num_ctx': 4096}

def build_context(results):
    context = ""
    for i, r in enumerate(results):
        context += f"\n--- Issue {i+1} ---\n"
        context += f"Title: {r['title']}\n"
        context += f"Label: {r['label']}\n"
        context += f"Details: {r['combined_text']}\n"
    return context

def build_prompt(query, context):
    return f"""You are an IT support assistant. Based on similar resolved issues below, give a concise resolution recommendation for the new issue.

NEW ISSUE: {query}

SIMILAR ISSUES:
{context}

Give a short, specific recommendation in 3-4 sentences."""

def dynamic_rag_ollama(query, top_k=3):
    results = dynamic_weighted_retrieve(query, top_k=top_k)
    context = build_context(results)
    response = ollama.chat(
        model='llama3.2:1b',
        options=GEN_OPTIONS,
        messages=[{'role': 'user', 'content': build_prompt(query, context)}]
    )
    return {
        'query': query,
        'retrieved_issues': results,
        'recommendation': response['message']['content']
    }

start = time.time()
output = dynamic_rag_ollama("window resize not working correctly")
print(f"Time taken: {round(time.time() - start, 2)} seconds")
print("\n=== GENERATED RECOMMENDATION ===")
print(output['recommendation'])

Time taken: 4.88 seconds

=== GENERATED RECOMMENDATION ===
Based on the similar resolved issues, I recommend trying the following:

I suggest checking the "View Logging" option in System Preferences > Mission Control (or Command + Shift + Left Arrow) to see if it's logging any errors or warnings that might be related to the issue. Additionally, I'll provide a specific recommendation for Issue 2: since the problem seems to be occurring with the newest macOS version, I recommend updating Rectangle to the latest version (0.67) and then testing the issue again. If the issue persists, it may be worth reaching out to the Rectangle support team for further assistance or troubleshooting steps.


In [11]:
import json

test_queries = [
    "window resize not working correctly",
    "authentication token expiring too quickly",
    "dark mode not applying to all components"
]

dynamic_results = {
    'configuration': 'dynamic_weighted',
    'generation_model': 'llama3.2:1b',
    'retrieval': 'flat index, semantic 0.5 + authority 0.3 + label 0.2',
    'results': []
}

for query in test_queries:
    print(f"Running: '{query}'")
    output = dynamic_rag_ollama(query)
    dynamic_results['results'].append({
        'query': output['query'],
        'retrieved_titles': [r['title'] for r in output['retrieved_issues']],
        'top_authors': [r['best_author'] for r in output['retrieved_issues']],
        'authority_scores': [r['author_score'] for r in output['retrieved_issues']],
        'recommendation': output['recommendation']
    })
    print("Done! ✅")

# Save results
with open('results/dynamic_results.json', 'w') as f:
    json.dump(dynamic_results, f, indent=2)

print("\nAll dynamic results saved! ✅")

Running: 'window resize not working correctly'
Done! ✅
Running: 'authentication token expiring too quickly'
Done! ✅
Running: 'dark mode not applying to all components'
Done! ✅

All dynamic results saved! ✅


In [12]:
import json
r = json.load(open('results/dynamic_results.json'))
print(r['results'][0]['retrieved_titles'])
print(r['results'][0]['top_authors'])

["Rectangle Doesn't Resize Windows", 'Dragging windows between screens makes them resize', 'Overlapping of the thirds ']
['rxhanson', 'rxhanson', 'rxhanson']


In [12]:
a = pd.read_csv('data/author_authority_scores.csv')
print(len(a.columns), '| ocornut resolutions:', a[a.author=='ocornut'].resolutions.values)

7 | ocornut resolutions: [92]


In [13]:
import os, pandas as pd
os.chdir('/Users/yaminiumarani/Desktop/rag_thesis_project')

a = pd.read_csv('data/author_authority_scores.csv')

print("=== TOP 5 BY UNIQUE ISSUES ===")
print(a.nlargest(5, 'unique_issues')[
    ['author', 'frequency', 'resolutions', 'unique_issues',
     'is_pure_reporter', 'authority_score']].to_string(index=False))
print(f"\nconsistency denominator (max unique_issues): {a['unique_issues'].max()}")

print("\n=== OCORNUT ===")
print(a[a.author == 'ocornut'].to_string(index=False))

df = pd.read_csv('data/github_issues_final.csv', low_memory=False)
mask = df[[f'answers_{i}_author' for i in range(6)]].eq('ocornut').any(axis=1)
print(f"\nissues where ocornut appears as a replier: {mask.sum()}")
print("repositories:")
print(df.loc[mask, 'repo_name'].value_counts().head())

=== TOP 5 BY UNIQUE ISSUES ===
        author  frequency  resolutions  unique_issues  is_pure_reporter  authority_score
       ocornut        296           92            132             False         1.000000
       sharkdp        218           62             98             False         0.713238
        lervag        184           71             86             False         0.690637
     Conduitry        118           45             83             False         0.503883
elasticmachine         97            2             83             False         0.295643

consistency denominator (max unique_issues): 132

=== OCORNUT ===
 author  frequency  resolutions  unique_issues  is_pure_reporter  consistency_score  authority_score
ocornut        296           92            132             False                1.0              1.0

issues where ocornut appears as a replier: 132
repositories:
repo_name
ocornut/imgui    132
Name: count, dtype: int64


In [14]:
import numpy as np
a = pd.read_csv('data/author_authority_scores.csv')

corrected = (0.3 * a.frequency / a.frequency.max()
           + 0.4 * a.resolutions / a.resolutions.max()
           + 0.3 * a.unique_issues / a.unique_issues.max())
corrected = np.where(a.is_pure_reporter, corrected * 0.2, corrected)
corrected = corrected / corrected.max()

a2 = a.assign(corrected=corrected)
print("Spearman rank correlation:", a.authority_score.corr(a2.corrected, method='spearman'))
print("\n=== TOP 10, CORRECTED ===")
print(a2.nlargest(10, 'corrected')[['author','authority_score','corrected']].to_string(index=False))
print("\n=== TOP 10, CURRENT ===")
print(a2.nlargest(10, 'authority_score')[['author','authority_score','corrected']].to_string(index=False))

Spearman rank correlation: 1.0

=== TOP 10, CORRECTED ===
             author  authority_score  corrected
            ocornut         1.000000   1.000000
            sharkdp         0.713238   0.713238
             lervag         0.690637   0.690637
magento-engcom-team         0.638345   0.638345
          Conduitry         0.503883   0.503883
          akien-mga         0.439098   0.439098
   oliviertassinari         0.387214   0.387214
            shilman         0.384474   0.384474
            gfrenoy         0.361086   0.361086
           rxhanson         0.350609   0.350609

=== TOP 10, CURRENT ===
             author  authority_score  corrected
            ocornut         1.000000   1.000000
            sharkdp         0.713238   0.713238
             lervag         0.690637   0.690637
magento-engcom-team         0.638345   0.638345
          Conduitry         0.503883   0.503883
          akien-mga         0.439098   0.439098
   oliviertassinari         0.387214   0.387214
     

In [15]:
import pandas as pd
df = pd.read_csv('data/github_issues_final.csv', low_memory=False)

mask = pd.concat(
    [(df[f'answers_{i}_author'] == 'ghost') & df[f'answers_{i}_body'].notna()
     for i in range(6)], axis=1).any(axis=1)

print("ghost unique issues (slots 0-5, body present):", mask.sum())
print("check:", 132 / mask.sum() if mask.sum() else None, "— should be ≈ 0.511628")

ghost unique issues (slots 0-5, body present): 258
check: 0.5116279069767442 — should be ≈ 0.511628


In [17]:
a = pd.read_csv('data/author_authority_scores.csv')
print("max frequency:    ", a.frequency.max())
print("max resolutions:  ", a.resolutions.max())
print("max unique_issues:", a.unique_issues.max())

max frequency:     296
max resolutions:   92
max unique_issues: 132


In [18]:
known = set(pd.read_csv('data/author_authority_scores.csv')['author'])
names = pd.concat([df[f'answers_{i}_author'].dropna() for i in range(6)])
print("reply slots with an author:", len(names))
print("not in table:", (~names.isin(known)).sum())
print(names[~names.isin(known)].value_counts().head())

reply slots with an author: 49284
not in table: 355
ghost    355
Name: count, dtype: int64


In [19]:
df = pd.read_csv('data/github_issues_final.csv')
assert 'combined_text' in df.columns      # confirms it's the final one

acols = [f'answers_{i}_author' for i in range(6)]
mask = df[acols].eq('elasticmachine')

print("slots:", mask.sum().sum(), "of", df[acols].notna().sum().sum())
print("issues touched:", mask.any(axis=1).sum(), "of", len(df))
print(mask.sum())

slots: 97 of 49284
issues touched: 83 of 15955
answers_0_author    77
answers_1_author    11
answers_2_author     4
answers_3_author     3
answers_4_author     1
answers_5_author     1
dtype: int64


In [21]:
aa = pd.read_csv('data/author_authority_scores.csv')   # adjust path
print(aa.columns.tolist())
print(aa[aa.iloc[:, 0] == 'elasticmachine'].to_string())

['author', 'frequency', 'resolutions', 'unique_issues', 'is_pure_reporter', 'consistency_score', 'authority_score']
            author  frequency  resolutions  unique_issues  is_pure_reporter  consistency_score  authority_score
25  elasticmachine         97            2             83             False           0.628788         0.295643


In [22]:
rep = pd.concat([df[f'answers_{i}_author'] for i in range(6)]).dropna()
print(rep.value_counts().head(25))

ghost                  355
ocornut                296
magento-engcom-team    230
sharkdp                218
lervag                 184
oliviertassinari       127
Conduitry              118
gfrenoy                102
akien-mga              100
Canop                   98
rxhanson                98
DaveDavenport           98
elasticmachine          97
borgmanJeremy           95
ckifer                  95
atomiks                 94
shilman                 93
bluebill1049            88
lauriejim               82
myspaghetti             82
posva                   81
Keats                   78
justinmayer             77
mbostock                76
fregante                75
Name: count, dtype: int64


In [23]:
m = df[[f'answers_{i}_author' for i in range(6)]].eq('magento-engcom-team')
for i in range(6):
    s = df.loc[m[f'answers_{i}_author'], f'answers_{i}_body']
    for v in s.head(4):
        print(repr(str(v)[:160]), '\n---')
print(aa[aa.author == 'magento-engcom-team'].to_string())

"@shavounet, thank you for your report.\nWe've created internal ticket(s) MAGETWO-83235 to track progress on the issue." 
---
'Hi @ihor-sviziev. Thank you for your report. \nTo help us process this issue please make sure that you provided the following information:\n- [x] Summary of the i' 
---
'Hi @apomili. Thank you for your report. \nTo help us process this issue please make sure that you provided the following information:\n- [x] Summary of the issue\n' 
---
'Hi @Den4ik. Thank you for your report. \nTo help us process this issue please make sure that you provided the following information:\n- [ ] Summary of the issue\n-' 
---
"@thdoan, thank you for your report.\nWe've created internal ticket(s) MAGETWO-70234 to track progress on the issue." 
---
'@shavounet, thank you for your report.\r\n\r\nUnfortunately, we are archiving this ticket now as it did not get much attention from both Magento Community and Core ' 
---
'Hi @engcom-backlog-nazar. Thank you for working on this issue. \nI

In [24]:
import json
r = json.load(open('results/dynamic_results.json'))   # adjust name
print(type(r), len(r))
print(json.dumps(r[0] if isinstance(r, list) else r, indent=2)[:1200])

<class 'dict'> 4
{
  "configuration": "dynamic_weighted",
  "generation_model": "llama3.2:1b",
  "retrieval": "flat index, semantic 0.5 + authority 0.3 + label 0.2",
  "results": [
    {
      "query": "window resize not working correctly",
      "retrieved_titles": [
        "Rectangle Doesn't Resize Windows",
        "Dragging windows between screens makes them resize",
        "Overlapping of the thirds "
      ],
      "top_authors": [
        "rxhanson",
        "rxhanson",
        "rxhanson"
      ],
      "authority_scores": [
        0.3506,
        0.3506,
        0.3506
      ],
      "recommendation": "Based on the similar resolved issues, here's a concise resolution recommendation for the new issue:\n\nTo resolve the window resize not working correctly issue, I recommend installing and upgrading to macOS 13.2.1 (22G140) or later, which includes improved window resizing capabilities that should address the overlapping of neighboring windows issue. Additionally, it is recomme

In [26]:
import json, glob

bots = {'elasticmachine', 'magento-engcom-team', 'ghost'}

for path in sorted(glob.glob('results/*results*.json')):
    r = json.load(open(path))
    recs = r.get('results', [])
    if not recs:
        print(path, '-- no results key'); continue

    keys = list(recs[0].keys())
    acols = [k for k in keys if 'author' in k.lower()]
    print(f"\n{path}")
    print("  keys:", keys)
    print("  depth:", {k: len(v) for k, v in recs[0].items() if isinstance(v, list)})

    if not acols:
        print("  no author field -- cannot check from this file")
        continue

    hits = []
    for q in recs:
        for col in acols:
            for a in q.get(col, []):
                if a in bots:
                    hits.append((a, col, str(q.get('query'))[:40]))
    print(f"  {len(recs)} queries -> {len(hits)} bot-authored candidates")
    for h in hits:
        print('    ', h)


results/baseline_results.json
  keys: ['query', 'retrieved_titles', 'recommendation']
  depth: {'retrieved_titles': 3}
  no author field -- cannot check from this file

results/dynamic_results.json
  keys: ['query', 'retrieved_titles', 'top_authors', 'authority_scores', 'recommendation']
  depth: {'retrieved_titles': 3, 'top_authors': 3, 'authority_scores': 3}
  3 queries -> 0 bot-authored candidates
results/evaluation_results.json -- no results key
results/guardrail_results.json -- no results key


In [27]:
import json
for nb in ['notebooks/04_dynamic_rag.ipynb', 'notebooks/06_partitioned_rag.ipynb']:
    for c in json.load(open(nb))['cells']:
        s = ''.join(c['source'])
        if 'label' in s.lower() and ('score' in s.lower() or 'partition' in s.lower()):
            print(nb, '\n', s, '\n---')

notebooks/04_dynamic_rag.ipynb 
 # Label relevance scoring
# Boosts issues whose labels match the query intent

label_keywords = {
    'bug': ['bug', 'error', 'crash', 'fail', 'broken', 'not working', 
            'incorrect', 'wrong', 'issue', 'problem'],
    'enhancement': ['feature', 'request', 'improve', 'add', 'support', 
                   'allow', 'enable', 'implement'],
    'documentation': ['doc', 'documentation', 'readme', 'guide', 
                     'tutorial', 'example', 'explain'],
    'good first issue': ['beginner', 'simple', 'easy', 'starter'],
    'help wanted': ['help', 'assist', 'support', 'need']
}

def get_label_relevance_score(query, label):
    if pd.isna(label):
        return 0.0
    
    query_lower = query.lower()
    label_lower = str(label).lower()
    
    # Direct label match in query
    if label_lower in query_lower:
        return 1.0
    
    # Check keyword matches
    for category, keywords in label_keywords.items():
        if category in label_